# Synteny Analysis and Collinear Blocks with JCVI (MCscan)

This notebook follows the official [MCscan (Python version) workflow](https://github.com/tanghaibao/jcvi/wiki/MCscan-(Python-version)) to identify and visualize collinear blocks between our Vellozia genomes (e.g. *V. peripherica* vs *V. tubiflora*).

**Prerequisites:**
- `jcvi` must be installed (e.g. via `jcviEnv`).
- We need the `.gff` (or `.bed`) and `.cds` (or `.pep`) files for both genomes.

*Make sure to run this notebook in the `jcviEnv` conda environment.*

## 1. Prepare Input Files

JCVI requires `.bed` files (with gene coordinates) and `.cds` (or `.pep`) files. If you have `.gff`, you can convert them using jcvi formats.

In [ ]:
# Example: Convert GFF to BED (Replace with your actual GFF/CDS paths)
# !python -m jcvi.formats.gff bed --type=mRNA --key=ID Vper.gff3 -o Vper.bed
# !python -m jcvi.formats.gff bed --type=mRNA --key=ID Vtub.gff3 -o Vtub.bed

# Make sure you also have Vper.cds and Vtub.cds in this directory or linked here.

## 2. Identify Orthologs / Synteny Blocks

Run the `jcvi.compara.catalog` module. This will perform LAST/DIAMOND alignments and identify syntenic blocks, generating `.anchors` and `.pdf` files.

In [ ]:
# Run ortholog finding
# This will output Vper.Vtub.anchors
# !python -m jcvi.compara.catalog ortholog Vper Vtub --no_strip_names

Check if the `.anchors` file was generated correctly:

In [ ]:
# !head -n 10 Vper.Vtub.anchors

## 3. Macro-Synteny Visualization (Karyotype)

To plot the karyotype macro-synteny, we need two configuration files: `seqids` and `layout`.

In [ ]:
# 1. Create seqids file (defines which chromosomes to plot)
# Replace with your actual chromosome IDs
seqids_content = """chr1A,chr2A,chr3A,chr4A,chr5A,chr6A,chr7A,chr8A,chr9A\nchr1A,chr2A,chr3A,chr4A,chr5A,chr6A,chr7A,chr8A,chr9A\n"""
with open("seqids", "w") as f:
    f.write(seqids_content)

# 2. Create layout file (defines how to plot them)
layout_content = """# y, xstart, xend, rotation, color, label, va,  bed
 .6,     .1,    .8,       0,      #ffb2b2, Vper, top, Vper.bed
 .4,     .1,    .8,       0,      #b2ffb2, Vtub, top, Vtub.bed
# edges
e, 0, 1, Vper.Vtub.anchors\n"""
with open("layout", "w") as f:
    f.write(layout_content)

In [ ]:
# Generate the Macro-synteny plot
# !python -m jcvi.graphics.karyotype seqids layout

## 4. Micro-Synteny Visualization (Local Blocks)

If you want to zoom in on specific collinear blocks (e.g. a specific OSCA gene region), we use `jcvi.graphics.synteny`.
We first need to extract the local block using `jcvi.compara.synteny screen` or subset the `.anchors` file.

In [ ]:
# Create a blocks file (example for one block)
# You should put the specific anchors you want to visualize in 'blocks'
# !head -n 50 Vper.Vtub.anchors > blocks

# Create layout for micro-synteny
layout_micro = """# x,   y, rotation,   ha,     va,   color, ratio,            label
0.5, 0.6,        0, left, center,       m,     1,       Vper_chr1
0.5, 0.4,        0, left, center, #fc8d62,     1,       Vtub_chr1
# edges
e, 0, 1\n"""
with open("blocks.layout", "w") as f:
    f.write(layout_micro)

In [ ]:
# Generate the Micro-synteny plot
# !python -m jcvi.graphics.synteny blocks Vper.bed Vper.bed blocks.layout